# IE-OPT-07 · 两仓三店辅料运输（PySCIPOpt 原创教学案例）

**场景**：澄湾辅料仓和石桥辅料仓本周要给三家门店补周转箱。箱子不能拆开，运费按箱计算。问每个仓往每家店发多少箱，总运费最小。

供需是平衡的：两个仓一共 11 箱，三家店一共要 11 箱。本例数据为合成教学数据，店名、仓名和运价都是为本站练习新编的。

**该 Notebook 是本地教学示例，不计入正式课程成绩。**

## 1. 模型要点

- 决策：非负整数 $x_{ij}$，表示仓 $i$ 发给店 $j$ 的箱数。
- 目标：最小化 $\sum_{i,j} c_{ij} x_{ij}$。
- 每个仓发出的箱数等于该仓库存；每家店收到的箱数等于该店需求。
- 平衡运输问题的约束矩阵是全幺模的，所以在整数供需下，线性规划的顶点最优解本身就是整数。这里仍把 $x_{ij}$ 声明为整数，和「按箱发货」一致。

这个模型不包含车辆往返、道路拥堵和缺货惩罚。

In [ ]:
from pyscipopt import Model, quicksum

warehouses = ["澄湾辅料仓", "石桥辅料仓"]
stores = ["杏林门店", "槐安门店", "汀洲门店"]
supply = [6, 5]
demand = [4, 3, 4]
cost = [
    [4, 7, 5],
    [6, 3, 8],
]
assert sum(supply) == sum(demand)

## 2. 求解并读出发货方案

最优状态不是 `optimal` 时不要把当前目标当成最小运费。

In [ ]:
def solve_transport(supply, demand, cost):
    model = Model("IE-OPT-07 box transport")
    model.hideOutput(True)
    x = {(i, j): model.addVar(vtype="I", lb=0, name=f"ship_{i}_{j}")
         for i in range(len(supply)) for j in range(len(demand))}
    for i in range(len(supply)):
        model.addCons(quicksum(x[i, j] for j in range(len(demand))) == supply[i])
    for j in range(len(demand)):
        model.addCons(quicksum(x[i, j] for i in range(len(supply))) == demand[j])
    model.setObjective(quicksum(cost[i][j] * x[i, j]
                                 for i in range(len(supply))
                                 for j in range(len(demand))), "minimize")
    model.optimize()
    status = str(model.getStatus())
    if status != "optimal":
        raise RuntimeError(f"Solver status is {status}; optimality not certified")
    shipments = [[int(round(model.getVal(x[i, j]))) for j in range(len(demand))]
                 for i in range(len(supply))]
    return {"objective": float(model.getObjVal()), "shipments": shipments}

result = solve_transport(supply, demand, cost)
print("最小运费:", result["objective"])
for i, name in enumerate(warehouses):
    sent = {stores[j]: result["shipments"][i][j] for j in range(len(stores))}
    print(name, sent)

## 3. 独立枚举

两个仓、三家店，把每个仓的箱子分出去的整数方案很少。枚举时检查店的需求是否刚好被满足，得到不依赖 SCIP 的最小运费。

In [ ]:
def oracle_min_cost(supply, demand, cost):
    best = [float("inf")]
    remaining = list(demand)

    def place(i, j, left, acc):
        if acc >= best[0]:
            return
        if i == len(supply):
            if all(value == 0 for value in remaining):
                best[0] = acc
            return
        if j == len(demand):
            if left == 0:
                nxt = supply[i + 1] if i + 1 < len(supply) else 0
                place(i + 1, 0, nxt, acc)
            return
        for qty in range(min(left, remaining[j]) + 1):
            remaining[j] -= qty
            place(i, j + 1, left - qty, acc + qty * cost[i][j])
            remaining[j] += qty

    place(0, 0, supply[0], 0)
    if best[0] == float("inf"):
        raise RuntimeError("no feasible shipment")
    return best[0]

oracle = oracle_min_cost(supply, demand, cost)
print("独立枚举最小运费:", oracle)
assert abs(result["objective"] - oracle) < 1e-6
for i in range(len(supply)):
    assert sum(result["shipments"][i]) == supply[i]
for j in range(len(demand)):
    assert sum(result["shipments"][i][j] for i in range(len(supply))) == demand[j]
print("通过：求解器目标与独立枚举一致，供需约束满足")

## 4. 可以接着改什么

1. 把某一格运价改高，看货流会不会绕开这条线。
2. 让总供给比总需求多 1 箱，把仓库约束改成「发出量不超过库存」，并给剩余库存一个持有费。
3. 这个静态发货方案不知道货车能不能在一天内跑完。要讨论路线，转到巡检回路和带容量配送那两份 Notebook。

**该 Notebook 是本地教学示例，不计入正式课程成绩。**